In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import glob
import os

In [2]:
df = pd.read_csv("../anac_data/Results_Corrected/all_fields_photometry_COMPLETE.csv")
print("Numbers of sources", len(df))
df.columns.tolist()

Numbers of sources 5769


['recno',
 'T17ID',
 'oldID',
 'RAJ2000',
 'DEJ2000',
 'Prob',
 'Rgc',
 'PA',
 'umag',
 'gmag',
 'rmag',
 'imag',
 'zmag',
 'e_umag',
 's_umag',
 'e_gmag',
 's_gmag',
 'e_rmag',
 's_rmag',
 'e_imag',
 's_imag',
 'e_zmag',
 's_zmag',
 'FIELD',
 'FLUX_F378_2',
 'FLUXERR_F378_2',
 'MAG_F378_2',
 'MAGERR_F378_2',
 'SNR_F378_2',
 'FLUX_F378_3',
 'FLUXERR_F378_3',
 'MAG_F378_3',
 'MAGERR_F378_3',
 'SNR_F378_3',
 'FLUX_F395_2',
 'FLUXERR_F395_2',
 'MAG_F395_2',
 'MAGERR_F395_2',
 'SNR_F395_2',
 'FLUX_F395_3',
 'FLUXERR_F395_3',
 'MAG_F395_3',
 'MAGERR_F395_3',
 'SNR_F395_3',
 'FLUX_F410_2',
 'FLUXERR_F410_2',
 'MAG_F410_2',
 'MAGERR_F410_2',
 'SNR_F410_2',
 'FLUX_F410_3',
 'FLUXERR_F410_3',
 'MAG_F410_3',
 'MAGERR_F410_3',
 'SNR_F410_3',
 'FLUX_F430_2',
 'FLUXERR_F430_2',
 'MAG_F430_2',
 'MAGERR_F430_2',
 'SNR_F430_2',
 'FLUX_F430_3',
 'FLUXERR_F430_3',
 'MAG_F430_3',
 'MAGERR_F430_3',
 'SNR_F430_3',
 'FLUX_F515_2',
 'FLUXERR_F515_2',
 'MAG_F515_2',
 'MAGERR_F515_2',
 'SNR_F515_2',
 'FLUX_F51

In [10]:
unique_sources = len(df.drop_duplicates(subset=['RAJ2000', 'DEJ2000']))
unique_sources

3209

In [11]:
df_unique = df.drop_duplicates(subset=['RAJ2000', 'DEJ2000'])

In [5]:
def seleccionar_buena_fotometria(df, apertura=2, umbral_azul=0.7, umbral_rojo=0.5):
    """
    Selecciona fuentes con buena fotometría basada en los errores
    
    Parameters:
    - df: DataFrame con los datos
    - apertura: 2 o 3 (para usar apertura 2 o 3 en los filtros JPCAM)
    - umbral_azul: umbral de error para filtros azules
    - umbral_rojo: umbral de error para filtros rojos
    """
    
    # Definir qué filtros son azules y cuáles rojos
    filtros_azules = ['umag', 'gmag', 'F378', 'F395', 'F410', 'F430']
    filtros_rojos = ['rmag', 'imag', 'zmag', 'F515','F660', 'F861']
    
    # Crear máscara inicial (todas True)
    mascara_buena_fot = pd.Series([True] * len(df), index=df.index)
    
    # Aplicar criterios para filtros azules
    for filtro in filtros_azules:
        if filtro in ['umag', 'gmag', 'rmag', 'imag', 'zmag']:
            # Para filtros ugriz usar columnas estándar
            col_error = f'e_{filtro}'
            if col_error in df.columns:
                mascara_buena_fot &= (df[col_error] < umbral_azul) & (df[col_error] > 0)
        else:
            # Para filtros JPCAM usar la apertura seleccionada
            col_error = f'MAGERR_{filtro}_{apertura}'
            if col_error in df.columns:
                mascara_buena_fot &= (df[col_error] < umbral_azul) & (df[col_error] > 0)
    
    # Aplicar criterios para filtros rojos
    for filtro in filtros_rojos:
        if filtro in ['rmag', 'imag', 'zmag']:
            # Para filtros ugriz usar columnas estándar
            col_error = f'e_{filtro}'
            if col_error in df.columns:
                mascara_buena_fot &= (df[col_error] < umbral_rojo) & (df[col_error] > 0)
        else:
            # Para filtros JPCAM usar la apertura seleccionada
            col_error = f'MAGERR_{filtro}_{apertura}'
            if col_error in df.columns:
                mascara_buena_fot &= (df[col_error] < umbral_rojo) & (df[col_error] > 0)
    
    return df[mascara_buena_fot].copy()


In [12]:
# Ejemplos de uso:

# Usar apertura 2 con umbrales por defecto
df_bueno_a2 = seleccionar_buena_fotometria(df_unique, apertura=2)
print(f"Fuentes con buena fotometría (apertura 2): {len(df_bueno_a2)}")

# Usar apertura 3 con umbrales por defecto
df_bueno_a3 = seleccionar_buena_fotometria(df_unique, apertura=3)
print(f"Fuentes con buena fotometría (apertura 3): {len(df_bueno_a3)}")

# Usar umbrales personalizados
df_bueno_personalizado = seleccionar_buena_fotometria(df_unique, apertura=2, umbral_azul=0.7, umbral_rojo=0.5)
print(f"Fuentes con buena fotometría (umbrales personalizados): {len(df_bueno_personalizado)}")


Fuentes con buena fotometría (apertura 2): 828
Fuentes con buena fotometría (apertura 3): 760
Fuentes con buena fotometría (umbrales personalizados): 828


In [13]:
# Save
df_bueno_a3.to_csv("../anac_data/Results_Corrected/all_fields_photometry_COMPLETE_high_quality.csv", index=False)

In [6]:
# Función adicional para comparar ambas aperturas
def comparar_aperturas(df, umbral_azul=0.5, umbral_rojo=0.3):
    """Compara cuántas fuentes pasan los criterios en cada apertura"""
    a2 = seleccionar_buena_fotometria(df, 2, umbral_azul, umbral_rojo)
    a3 = seleccionar_buena_fotometria(df, 3, umbral_azul, umbral_rojo)
    
    print(f"Apertura 2: {len(a2)} fuentes")
    print(f"Apertura 3: {len(a3)} fuentes")
    
    # Fuentes que pasan en AMBAS aperturas
    comunes = set(a2['recno']).intersection(set(a3['recno']))
    print(f"Fuentes en ambas aperturas: {len(comunes)}")
    
    return a2, a3

# Comparar
a2, a3 = comparar_aperturas(df)

Apertura 2: 70 fuentes
Apertura 3: 68 fuentes
Fuentes en ambas aperturas: 59
